# 09 — Backtest de origem móvel

Toda a avaliação até aqui repousa sobre **um único corte**: teste em 30 meses de 2022-01 a 2024-06. Isso traz duas limitações:

1. **Poder estatístico.** O Diebold-Mariano do notebook 04 dispõe de apenas `T = 30` observações
   mensais.
2. **Um único regime.** O período testado é dominado pelo aperto monetário de 2022–23. Nada garante
   que o resultado se sustente na recessão de 2015–16, na recuperação de 2017–19 ou na COVID.

O **backtest de origem móvel** ataca as duas: em vez de um corte, desliza-se o ponto de previsão ao
longo de toda a história, reajustando o modelo periodicamente.

```
refit 2016-01: treino até 2015-07 │emb│ prevê 2016-01, 02, 03  (alvo em t+6)
refit 2016-04: treino até 2015-10 │emb│ prevê 2016-04, 05, 06
      ⋮                  ⋮                        ⋮
refit 2023-10: treino até 2023-04 │emb│ prevê 2023-10, 11, 12
```

## Protocolo

| Parâmetro | Escolha | Justificativa |
|---|---|---|
| Janela | **Expansiva** | Usa toda a história disponível a cada origem |
| Primeira origem | **2016-01** | Deixa 3 anos de treino inicial |
| Última origem | **2024-06** | O alvo em `t+6` precisa ser observável |
| Reajuste | **Trimestral** | Padrão de recalibração; 32 reajustes cobrindo 96 meses |
| Embargo | **horizonte + defasagem** (8 meses), a cada origem | Idêntico ao dos notebooks anteriores |
| Variáveis | Conjunto **estendido** do notebook 08 | Inclui as variáveis adicionais do SCR |
| Limiar do evento | Recalculado **em cada janela de treino** | Evita usar informação futura na definição do alvo |

## Baselines

Para o **nível**: persistência (`ŷ = taxa(t)`) — o adversário que se mostrou binding.
Para o **evento**: `vencido_lag1` e nível atual — os melhores baselines do notebook 07.

> O ARIMA foi deixado de fora deste backtest deliberadamente. Sua implementação de origem móvel
> estima os parâmetros uma única vez e depois apenas atualiza o estado; mantê-lo por oito anos sem
> reestimação, enquanto os modelos de ML são reajustados a cada trimestre, seria uma comparação
> desfavorável a ele. A comparação com o ARIMA está no notebook 04, sob corte único e condições
> equivalentes.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score
from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings('ignore')

BASE = Path().resolve().parent
PROC = BASE / 'data' / 'processed'
SEG_KEYS = ['cnae_secao', 'porte', 'modalidade']

with open(PROC / 'ml_metadata_ext.json', encoding='utf-8') as f:
    mx = json.load(f)
FEATS = mx['feature_cols_ext']
HORIZONTE = int(mx['horizonte'])
Q_LIMIAR  = float(mx['quantil_limiar'])
# Defasagem de publicacao (revisao A1): embargo = horizonte + defasagem; o nivel conhecido na
# origem e `nivel_col` (= nivel_disp quando ha defasagem).
LAG_SCR   = int(mx.get('lag_scr', 0))
EMBARGO   = int(mx.get('embargo_meses', HORIZONTE))
NIVEL     = mx.get('nivel_col', 'taxa_inadimplencia')

df = (pd.read_parquet(PROC / 'ml_dataset_ext.parquet')
        .sort_values('data_base').reset_index(drop=True))

meses = pd.to_datetime(pd.Series(df['data_base'].unique())).sort_values().tolist()
EVAL_INI = pd.Timestamp('2016-01-01')
meses_eval = [m for m in meses if m >= EVAL_INI]
PASSO = 3                                   # reajuste trimestral
refits = list(range(0, len(meses_eval), PASSO))

print(f'Variaveis: {len(FEATS)} (conjunto estendido)')
print(f'Meses de avaliacao: {len(meses_eval)} '
      f'({meses_eval[0].date()} -> {meses_eval[-1].date()})')
print(f'Reajustes (trimestrais): {len(refits)}')
print(f'Embargo: {EMBARGO} meses a cada origem '
      f'(horizonte {HORIZONTE} + defasagem de publicacao {LAG_SCR})  |  nivel conhecido: {NIVEL}')

## 1. Execução do backtest

A cada reajuste: treina com tudo cujo alvo já é observável (`data_base ≤ origem − embargo`),
recalcula o limiar do evento na janela de treino e prevê os três meses seguintes.

In [ ]:
def modelos():
    imp = lambda: SimpleImputer(strategy='median')
    return {
        'reg': {
            'RandomForest': Pipeline([('imp', imp()),
                ('m', RandomForestRegressor(n_estimators=200, max_depth=12,
                                            min_samples_leaf=10, n_jobs=-1, random_state=42))]),
            'XGBoost': Pipeline([('imp', imp()),
                ('m', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                   subsample=0.8, colsample_bytree=0.8,
                                   random_state=42, verbosity=0))]),
        },
        'clf': {
            'RandomForest': Pipeline([('imp', imp()),
                ('m', RandomForestClassifier(n_estimators=200, max_depth=12,
                                             min_samples_leaf=10, n_jobs=-1, random_state=42))]),
            'XGBoost': Pipeline([('imp', imp()),
                ('m', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                    subsample=0.8, colsample_bytree=0.8, random_state=42,
                                    verbosity=0, eval_metric='logloss'))]),
        },
    }

registros = []
for k, i in enumerate(refits, 1):
    origem = meses_eval[i]
    # so entra no treino a linha cujo alvo (d + h) ja estava PUBLICADO na origem
    lim_treino = origem - pd.DateOffset(months=EMBARGO)
    tr = df[df['data_base'] <= lim_treino]
    if len(tr) < 3000:
        print(f'  [{k:2d}/{len(refits)}] {origem.date()} treino insuficiente ({len(tr)}), pulado')
        continue

    # Limiar do evento recalculado NA JANELA DE TREINO (sem informacao futura)
    limiar = float(tr['delta6'].quantile(Q_LIMIAR))
    y_tr_ev = (tr['delta6'] > limiar).astype(int).to_numpy()
    y_tr_lv = tr['target'].to_numpy()
    X_tr = tr[FEATS]

    mods = modelos()
    ajust = {'reg': {}, 'clf': {}}
    for nome, p in mods['reg'].items():
        ajust['reg'][nome] = p.fit(X_tr, y_tr_lv)
    for nome, p in mods['clf'].items():
        ajust['clf'][nome] = p.fit(X_tr, y_tr_ev)

    for j in range(i, min(i + PASSO, len(meses_eval))):
        mes = meses_eval[j]
        te = df[df['data_base'] == mes]
        if len(te) == 0:
            continue
        # Revisao A3: as chaves do segmento e a carteira ativa (valor real em t) sao gravadas
        # para permitir estratificar e ponderar a avaliacao; nao sao variaveis do modelo.
        r = pd.DataFrame({
            'data_base': mes,
            'origem_refit': origem,
            'limiar': limiar,
            'cnae_secao': te['cnae_secao'].to_numpy(),
            'porte': te['porte'].to_numpy(),
            'modalidade': te['modalidade'].to_numpy(),
            'carteira_ativa': (te['carteira_ativa'].to_numpy()
                               if 'carteira_ativa' in te.columns else np.nan),
            'y_nivel': te['target'].to_numpy(),
            'y_evento': (te['delta6'] > limiar).astype(int).to_numpy(),
            'Persistencia': te[NIVEL].to_numpy(),      # ultimo valor CONHECIDO na origem
            'Vencido t-1': te['vencido_lag1'].to_numpy(),
            'Nivel atual': te[NIVEL].to_numpy(),
        })
        if 'vol12_disp' in te.columns:                  # referencial de volatilidade (A2a)
            r['Volatilidade 12m'] = te['vol12_disp'].to_numpy()
        for nome, m in ajust['reg'].items():
            r[f'reg_{nome}'] = m.predict(te[FEATS])
        for nome, m in ajust['clf'].items():
            r[f'clf_{nome}'] = m.predict_proba(te[FEATS])[:, 1]
        registros.append(r)

    print(f'  [{k:2d}/{len(refits)}] {origem.date()}  treino={len(tr):>6,}  '
          f'limiar={limiar*100:5.2f} p.p.')

bt = pd.concat(registros, ignore_index=True)
bt.to_parquet(PROC / 'backtest_predicoes.parquet', index=False)
print()
print(f'Backtest concluido: {len(bt):,} previsoes em {bt.data_base.nunique()} meses')
print(f'Taxa-base de eventos no periodo: {bt.y_evento.mean():.2%}')
print('Salvo: backtest_predicoes.parquet')

## 2. Desempenho agregado no backtest

In [ ]:
y_lv, y_ev = bt['y_nivel'].to_numpy(), bt['y_evento'].to_numpy()

lin = []
for nome in ['RandomForest', 'XGBoost']:
    p = bt[f'reg_{nome}'].to_numpy()
    lin.append({'Alvo': 'nivel', 'Preditor': nome,
                'RMSE': np.sqrt(mean_squared_error(y_lv, p)),
                'MAE': mean_absolute_error(y_lv, p)})
p = bt['Persistencia'].to_numpy()
lin.append({'Alvo': 'nivel', 'Preditor': 'Persistencia',
            'RMSE': np.sqrt(mean_squared_error(y_lv, p)),
            'MAE': mean_absolute_error(y_lv, p)})

nivel = pd.DataFrame(lin).sort_values('RMSE').round(4)
print('=== Alvo NIVEL (RMSE/MAE no backtest completo) ===')
print(nivel.to_string(index=False))

BASELINES_EV = ['Vencido t-1', 'Nivel atual'] + (['Volatilidade 12m'] if 'Volatilidade 12m' in bt.columns else [])
lin2 = []
for nome in ['RandomForest', 'XGBoost']:
    s = bt[f'clf_{nome}'].to_numpy()
    top = np.argsort(-s)[:int(len(s) * 0.10)]
    lin2.append({'Alvo': 'evento', 'Preditor': nome, 'AUC': roc_auc_score(y_ev, s),
                 'P@10%': y_ev[top].mean(), 'Lift': y_ev[top].mean() / y_ev.mean()})
for nome in BASELINES_EV:
    s = bt[nome].to_numpy(dtype=float)
    ok = np.isfinite(s)
    top = np.argsort(-np.where(ok, s, -np.inf))[:int(ok.sum() * 0.10)]
    lin2.append({'Alvo': 'evento', 'Preditor': nome, 'AUC': roc_auc_score(y_ev[ok], s[ok]),
                 'P@10%': y_ev[top].mean(), 'Lift': y_ev[top].mean() / y_ev.mean()})

evento = pd.DataFrame(lin2).sort_values('AUC', ascending=False).round(4)
print()
print('=== Alvo EVENTO (AUC no backtest completo) ===')
print(evento.to_string(index=False))

pd.concat([nivel, evento]).to_csv(PROC / 'resultados_backtest.csv', index=False)
print()
print('Salvo: resultados_backtest.csv')

## 3. Significância com a amostra ampliada

A pergunta central: com ~96 origens em vez de 30, o ganho do ML sobre a persistência no **nível**
passa a ser estatisticamente significativo?

In [ ]:
from scipy import stats as sps

def dm_test_panel(dates, err_a, err_b, h=6, loss='se'):
    """Diebold-Mariano em painel: agrega o diferencial de perda por mes e testa a serie
    mensal com variancia Newey-West (lag h-1) e correcao HLN. d>0 => modelo B melhor."""
    d = err_a**2 - err_b**2 if loss == 'se' else np.abs(err_a) - np.abs(err_b)
    s = pd.Series(d).groupby(pd.Series(dates).values).mean().sort_index()
    T = len(s); dbar = float(s.mean()); u = (s - dbar).to_numpy()
    lrv = float(u @ u) / T
    L = h - 1
    for lag in range(1, min(L, T - 1) + 1):
        lrv += 2 * (1 - lag / (L + 1)) * float(u[lag:] @ u[:-lag]) / T
    dm = dbar / np.sqrt(max(lrv, 1e-16) / T)
    hln = np.sqrt(max((T + 1 - 2 * h + h * (h - 1) / T) / T, 1e-12))
    dm_hln = dm * hln
    return {'T_meses': T, 'DM_HLN': round(dm_hln, 3),
            'p_valor': round(2 * (1 - sps.t.cdf(abs(dm_hln), df=T - 1)), 4)}

datas = bt['data_base'].to_numpy()
err_p = y_lv - bt['Persistencia'].to_numpy()
out = []
for nome in ['RandomForest', 'XGBoost']:
    err_m = y_lv - bt[f'reg_{nome}'].to_numpy()
    for loss, rot in [('se', 'quadratica (RMSE)'), ('ae', 'absoluta (MAE)')]:
        r = dm_test_panel(datas, err_p, err_m, h=HORIZONTE, loss=loss)
        out.append({'Comparacao': f'Persistencia vs {nome}', 'Perda': rot, **r})

dm = pd.DataFrame(out)
print('=== Diebold-Mariano no backtest (d>0 e p<0.05 => o modelo de ML e melhor) ===')
print(dm.to_string(index=False))
dm.to_csv(PROC / 'resultados_backtest_dm.csv', index=False)
print()
print('Salvo: resultados_backtest_dm.csv')

In [ ]:
def boot_mensal_auc(y, s1, s2, meses_arr, B=600, seed=42):
    rng = np.random.default_rng(seed)
    uniq = np.unique(meses_arr)
    idx = {m: np.where(meses_arr == m)[0] for m in uniq}
    difs = []
    for _ in range(B):
        sel = rng.choice(uniq, size=len(uniq), replace=True)
        ii = np.concatenate([idx[m] for m in sel])
        yy = y[ii]
        if yy.min() == yy.max():
            continue
        a, b = s1[ii], s2[ii]
        ok = np.isfinite(a) & np.isfinite(b)
        difs.append(roc_auc_score(yy[ok], a[ok]) - roc_auc_score(yy[ok], b[ok]))
    difs = np.array(difs)
    return np.percentile(difs, 2.5), np.percentile(difs, 97.5), (difs <= 0).mean()

s_venc = bt['Vencido t-1'].to_numpy()
out2 = []
for nome in ['RandomForest', 'XGBoost']:
    s_m = bt[f'clf_{nome}'].to_numpy()
    ok = np.isfinite(s_venc) & np.isfinite(s_m)
    lo, hi, pb = boot_mensal_auc(y_ev[ok], s_m[ok], s_venc[ok], datas[ok])
    out2.append({'Comparacao': f'{nome} vs Vencido t-1',
                 'AUC modelo': round(roc_auc_score(y_ev[ok], s_m[ok]), 4),
                 'AUC baseline': round(roc_auc_score(y_ev[ok], s_venc[ok]), 4),
                 'IC95 boot': f'[{lo:+.4f}, {hi:+.4f}]', 'p (boot)': round(pb, 4)})

print('=== Evento: ganho sobre o melhor baseline (bootstrap de bloco mensal) ===')
print(pd.DataFrame(out2).to_string(index=False))
pd.DataFrame(out2).to_csv(PROC / 'resultados_backtest_auc.csv', index=False)
print()
print('Salvo: resultados_backtest_auc.csv')

In [ ]:
# === Revisao A3: a hipotese da cauda — erro por quartil de volatilidade e de carteira ===
# Se o ganho do ML sobre a persistencia vem dos "erros grandes", ele deve concentrar-se nos
# segmentos mais volateis. E se esses segmentos forem os de carteira pequena, o ganho pode ter
# pouca relevancia economica: por isso tambem se reporta o erro PONDERADO pela carteira ativa.
PREDS_NV = [('RF', 'reg_RandomForest'), ('XGB', 'reg_XGBoost'), ('Persist.', 'Persistencia')]
lin3 = []
for crit, col in [('Volatilidade 12m', 'Volatilidade 12m'), ('Carteira ativa', 'carteira_ativa')]:
    if col not in bt.columns or bt[col].isna().all():
        print(f'coluna {col} ausente no backtest — quartis por {crit} nao calculados')
        continue
    v = bt[col].astype(float)
    q = pd.qcut(v.rank(method='first'), 4, labels=['Q1 (menor)', 'Q2', 'Q3', 'Q4 (maior)'])
    for lab in q.cat.categories:
        msk = (q == lab).to_numpy()
        g = bt[msk]
        yl = g['y_nivel'].to_numpy()
        d = {'Criterio': crit, 'Quartil': lab, 'n': int(len(g)),
             'valor medio': float(v[msk].mean())}
        for rot, pc in PREDS_NV:
            d[f'RMSE {rot}'] = round(float(np.sqrt(mean_squared_error(yl, g[pc]))), 4)
            d[f'MAE {rot}']  = round(float(mean_absolute_error(yl, g[pc])), 4)
        d['ganho RMSE RF'] = f"{(d['RMSE Persist.'] - d['RMSE RF']) / d['RMSE Persist.'] * 100:+.1f}%"
        d['ganho MAE RF']  = f"{(d['MAE Persist.'] - d['MAE RF']) / d['MAE Persist.'] * 100:+.1f}%"
        lin3.append(d)

# Erro ponderado pela carteira ativa (relevancia economica)
if 'carteira_ativa' in bt.columns and bt['carteira_ativa'].notna().any():
    w = bt['carteira_ativa'].to_numpy(dtype=float); okw = np.isfinite(w) & (w > 0)
    d = {'Criterio': 'Ponderado por carteira', 'Quartil': 'todos', 'n': int(okw.sum()),
         'valor medio': float(w[okw].mean())}
    for rot, pc in PREDS_NV:
        e = y_lv[okw] - bt[pc].to_numpy(dtype=float)[okw]
        d[f'RMSE {rot}'] = round(float(np.sqrt(np.average(e ** 2, weights=w[okw]))), 4)
        d[f'MAE {rot}']  = round(float(np.average(np.abs(e), weights=w[okw])), 4)
    d['ganho RMSE RF'] = f"{(d['RMSE Persist.'] - d['RMSE RF']) / d['RMSE Persist.'] * 100:+.1f}%"
    d['ganho MAE RF']  = f"{(d['MAE Persist.'] - d['MAE RF']) / d['MAE Persist.'] * 100:+.1f}%"
    lin3.append(d)

quartis = pd.DataFrame(lin3)
print('=== Erro no NIVEL por quartil (backtest 2016-2024) ===')
print(quartis.to_string(index=False))
quartis.to_csv(PROC / 'resultados_backtest_quartis.csv', index=False)
print()
print('Salvo: resultados_backtest_quartis.csv')

In [ ]:
# === Revisao A2c no backtest: AUC media por mes de origem (ordenacao DENTRO do mes) ===
# IC bootstrap (600 reamostragens de meses, diferencas pareadas) da vantagem sobre o credito vencido.
def aucs_por_mes_bt(y, s, meses, min_n=20):
    s = np.asarray(s, dtype=float)
    out = {}
    for m in np.unique(meses):
        ii = meses == m
        yy, ss = y[ii], s[ii]
        ok = np.isfinite(ss)
        if ok.sum() < min_n or yy[ok].min() == yy[ok].max():
            continue
        out[m] = roc_auc_score(yy[ok], ss[ok])
    return out

def ic_dif_mensal_bt(a_mod, a_base, B=600, seed=42):
    comuns = sorted(set(a_mod) & set(a_base))
    d = np.array([a_mod[m] - a_base[m] for m in comuns])
    rng = np.random.default_rng(seed)
    bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(B)])
    return float(d.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

preditores_ev = [('RandomForest', 'clf_RandomForest'), ('XGBoost', 'clf_XGBoost')] + \
                [(b, b) for b in BASELINES_EV]
por_mes_bt = {rot_: aucs_por_mes_bt(y_ev, bt[col].to_numpy(dtype=float), datas)
              for rot_, col in preditores_ev}
base_bt = por_mes_bt['Vencido t-1']
lin5 = []
for rot_, col in preditores_ev:
    s = bt[col].to_numpy(dtype=float); ok = np.isfinite(s)
    vals = np.array(list(por_mes_bt[rot_].values()))
    linha = {'Preditor': rot_, 'AUC agrupada': round(roc_auc_score(y_ev[ok], s[ok]), 4),
             'AUC media mensal': round(float(vals.mean()), 4),
             'dp mensal': round(float(vals.std(ddof=1)), 4), 'meses': int(len(vals))}
    if rot_ != 'Vencido t-1':
        dm_, lo, hi = ic_dif_mensal_bt(por_mes_bt[rot_], base_bt)
        linha['dif vs vencido'] = round(dm_, 4)
        linha['IC95 dif vs vencido'] = f'[{lo:+.4f}, {hi:+.4f}]'
    lin5.append(linha)
df_aucm_bt = pd.DataFrame(lin5).sort_values('AUC media mensal', ascending=False)
print('=== Backtest: AUC agrupada vs AUC media por mes (2016-2024); IC bootstrap de meses ===')
print(df_aucm_bt.to_string(index=False))
df_aucm_bt.to_csv(PROC / 'resultados_backtest_auc_mensal.csv', index=False)
print()
print('Salvo: resultados_backtest_auc_mensal.csv')

## 4. Desempenho ao longo do tempo

A figura central deste notebook: como o desempenho evolui através de cinco regimes distintos
(recessão de 2015–16, recuperação de 2017–19, COVID em 2020, aperto monetário de 2021–23).

In [ ]:
# Metricas por mes de origem
por_mes = []
for mes, g in bt.groupby('data_base'):
    yl, ye = g['y_nivel'].to_numpy(), g['y_evento'].to_numpy()
    d = {'data_base': mes}
    for nome in ['RandomForest', 'XGBoost']:
        d[f'RMSE {nome}'] = np.sqrt(mean_squared_error(yl, g[f'reg_{nome}']))
        if ye.min() != ye.max():
            d[f'AUC {nome}'] = roc_auc_score(ye, g[f'clf_{nome}'])
    d['RMSE Persistencia'] = np.sqrt(mean_squared_error(yl, g['Persistencia']))
    sv = g['Vencido t-1'].to_numpy(); ok = np.isfinite(sv)
    if ye[ok].min() != ye[ok].max():
        d['AUC Vencido t-1'] = roc_auc_score(ye[ok], sv[ok])
    por_mes.append(d)

pm = pd.DataFrame(por_mes).set_index('data_base').sort_index()
pm.to_csv(PROC / 'resultados_backtest_por_mes.csv')

COR = {'RandomForest': '#2a78d6', 'XGBoost': '#1baf7a',
       'Persistencia': '#008300', 'Vencido t-1': '#008300'}
ROTULO = {'Persistencia': 'Persistencia (ultimo publ.)', 'Vencido t-1': 'Vencido (ultimo publ.)'}
SURF, GRID, AXIS = '#fcfcfb', '#e1e0d9', '#c3c2b7'
INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'
JAN = 6      # media movel de 6 meses, para reduzir ruido

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True, facecolor=SURF)

ax = axes[0]; ax.set_facecolor(SURF)
for nome in ['RandomForest', 'XGBoost', 'Persistencia']:
    serie = pm[f'RMSE {nome}'].rolling(JAN, min_periods=3).mean()
    ax.plot(serie.index, serie, color=COR[nome], linewidth=2,
            linestyle='--' if nome == 'Persistencia' else '-', label=ROTULO.get(nome, nome))
ax.set_ylabel('RMSE (média móvel 6m)', fontsize=10, color=INK2)
ax.set_title('Erro de previsão do NÍVEL ao longo do backtest', fontsize=12, color=INK, pad=10)
ax.legend(fontsize=9, frameon=False, labelcolor=INK2)
ax.grid(True, color=GRID, linewidth=0.7)

ax = axes[1]; ax.set_facecolor(SURF)
for nome in ['RandomForest', 'XGBoost', 'Vencido t-1']:
    col = f'AUC {nome}'
    if col not in pm:
        continue
    serie = pm[col].rolling(JAN, min_periods=3).mean()
    ax.plot(serie.index, serie, color=COR[nome], linewidth=2,
            linestyle='--' if nome == 'Vencido t-1' else '-', label=ROTULO.get(nome, nome))
ax.axhline(0.5, color=AXIS, linewidth=1, linestyle=':')
ax.set_ylabel('AUC (média móvel 6m)', fontsize=10, color=INK2)
ax.set_xlabel('Mês de origem da previsão', fontsize=10, color=INK2)
ax.set_title('Detecção de EVENTOS de deterioração ao longo do backtest',
             fontsize=12, color=INK, pad=10)
ax.legend(fontsize=9, frameon=False, labelcolor=INK2)
ax.grid(True, color=GRID, linewidth=0.7)

for a in axes:
    for s_ in ['top', 'right']:
        a.spines[s_].set_visible(False)
    for s_ in ['left', 'bottom']:
        a.spines[s_].set_color(AXIS)
    a.tick_params(colors=MUTED, labelsize=9)

plt.tight_layout()
plt.savefig(PROC / 'resultados_backtest_tempo.png', dpi=150,
            bbox_inches='tight', facecolor=SURF)
plt.show()
print('Salvo: resultados_backtest_tempo.png / resultados_backtest_por_mes.csv')

## 5. Desempenho por regime macroeconômico

In [ ]:
# Revisao C7: o backtest tem origens ate 2024-06; a tabela por regime passa a cobri-las.
REGIMES = [('2016-2017  recessao e saida', '2016-01-01', '2017-12-31'),
           ('2018-2019  recuperacao',     '2018-01-01', '2019-12-31'),
           ('2020-2021  COVID',           '2020-01-01', '2021-12-31'),
           ('2022-2023  aperto monetario','2022-01-01', '2023-12-31'),
           ('2024  jan-jun',              '2024-01-01', '2024-06-30')]

lin = []
for rot, ini, fim in REGIMES:
    g = bt[(bt.data_base >= ini) & (bt.data_base <= fim)]
    if len(g) == 0:
        continue
    yl, ye = g['y_nivel'].to_numpy(), g['y_evento'].to_numpy()
    rmse_rf = np.sqrt(mean_squared_error(yl, g['reg_RandomForest']))
    rmse_ps = np.sqrt(mean_squared_error(yl, g['Persistencia']))
    sv = g['Vencido t-1'].to_numpy(); ok = np.isfinite(sv)
    lin.append({'Regime': rot, 'meses': g.data_base.nunique(),
                'eventos': f'{ye.mean():.1%}',
                'RMSE RF': round(rmse_rf, 4), 'RMSE Persist.': round(rmse_ps, 4),
                'ganho RMSE': f'{(rmse_ps - rmse_rf) / rmse_ps * 100:+.1f}%',
                'AUC RF': round(roc_auc_score(ye, g['clf_RandomForest']), 4),
                'AUC XGB': round(roc_auc_score(ye, g['clf_XGBoost']), 4),
                'AUC Vencido': round(roc_auc_score(ye[ok], sv[ok]), 4)})

reg = pd.DataFrame(lin)
print('=== Desempenho por regime macroeconomico ===')
print(reg.to_string(index=False))
reg.to_csv(PROC / 'resultados_backtest_regimes.csv', index=False)
print()
print('Salvo: resultados_backtest_regimes.csv')